In [1]:
!pip install -q gdown gradio opencv-python Pillow torch torchvision

In [2]:
import os
import cv2
import json
import gdown
import tempfile
import requests
import traceback
import numpy as np
from PIL import Image
import gradio as gr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NOTUMOR_CONFIDENCE_FLOOR = 0.45

CLASS_SEG_PROFILES = {
    "Глиома": {
        "cam_l4_weight": 0.55,
        "cam_l3_weight": 0.45,
        "skull_erode_ratio": 0.048,
        "max_area_ratio": 0.26,
        "z_bright_thresh": 0.62,
        "z_dark_thresh": -0.52
    },
    "Менингиома": {
        "cam_l4_weight": 0.65,
        "cam_l3_weight": 0.35,
        "otsu_multiplier": 0.86,
        "min_thresh": 0.33,
        "skull_erode_ratio": 0.010,
        "max_area_ratio": 0.32
    },
    "Гипофиз": {
        "cam_l4_weight": 0.60,
        "cam_l3_weight": 0.40,
        "otsu_multiplier": 0.92,
        "min_thresh": 0.42,
        "skull_erode_ratio": 0.040,
        "max_area_ratio": 0.055
    }
}

WEIGHTS_FILE = "best_resnet50_latest.pth"
if not os.path.exists(WEIGHTS_FILE):
    gdown.download("https://drive.google.com/uc?id=18C2quutj5WNqhHAfSm_UKlxlqcNyuid9", WEIGHTS_FILE, quiet=False)

ckpt = torch.load(WEIGHTS_FILE, map_location=device, weights_only=False)
if isinstance(ckpt, dict) and 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

num_classes = ckpt['fc.1.weight'].shape[0] if 'fc.1.weight' in ckpt else ckpt['fc.weight'].shape[0]
use_dropout_fc = 'fc.1.weight' in ckpt

model_diag = models.resnet50(weights=None)
if use_dropout_fc:
    model_diag.fc = nn.Sequential(nn.Dropout(0.30), nn.Linear(model_diag.fc.in_features, num_classes))
else:
    model_diag.fc = nn.Linear(model_diag.fc.in_features, num_classes)

model_diag.load_state_dict(ckpt)
model_diag = model_diag.to(device).eval()

use_gatekeeper = False
if os.path.exists("best_gatekeeper.pth"):
    try:
        model_gate = models.mobilenet_v3_small(weights=None)
        model_gate.classifier[3] = nn.Linear(model_gate.classifier[3].in_features, 2)
        model_gate.load_state_dict(torch.load("best_gatekeeper.pth", map_location=device, weights_only=False))
        model_gate = model_gate.to(device).eval()
        use_gatekeeper = True
    except Exception:
        use_gatekeeper = False


class AdaptiveDualGradCAMPlusPlus:
    def __init__(self, model):
        self.model = model
        self.l4_grad, self.l4_act = None, None
        self.l3_grad, self.l3_act = None, None
        self.model.layer4[-1].register_forward_hook(lambda m, i, o: setattr(self, 'l4_act', o.detach()))
        self.model.layer4[-1].register_full_backward_hook(lambda m, gi, go: setattr(self, 'l4_grad', go[0].detach()))
        self.model.layer3[-1].register_forward_hook(lambda m, i, o: setattr(self, 'l3_act', o.detach()))
        self.model.layer3[-1].register_full_backward_hook(lambda m, gi, go: setattr(self, 'l3_grad', go[0].detach()))

    def _compute_cam_pp(self, grads, acts):
        if grads is None or acts is None:
            return np.zeros((16, 16), dtype=np.float32)
        grads_pow2 = grads.pow(2)
        grads_pow3 = grads.pow(3)
        sum_acts = acts.sum(dim=(2, 3), keepdim=True)
        aij = torch.where(grads > 0, grads_pow2 / (2.0 * grads_pow2 + sum_acts * grads_pow3 + 1e-6), torch.zeros_like(grads))
        weights = (torch.clamp(grads, min=0.0) * aij).sum(dim=(2, 3), keepdim=True)
        cam = np.maximum((weights * acts).sum(dim=1).squeeze().detach().cpu().numpy(), 0)
        return (cam / cam.max()) if cam.max() > 0 else cam

    def get_cam(self, tensor_img, class_idx, w_l4=0.55, w_l3=0.45):
        self.model.zero_grad()
        output = self.model(tensor_img)
        output[0, class_idx].backward()
        cam4 = self._compute_cam_pp(self.l4_grad, self.l4_act)
        cam3 = self._compute_cam_pp(self.l3_grad, self.l3_act)
        cam4_res = cv2.resize(cam4, (cam3.shape[1], cam3.shape[0]))
        fused = np.maximum(w_l4 * cam4_res + w_l3 * cam3, 0)
        return (fused / fused.max()) if fused.max() > 0 else fused

grad_cam_engine = AdaptiveDualGradCAMPlusPlus(model_diag)


def safe_pil_to_tensor(pil_img):
    arr = np.array(pil_img, dtype=np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    return torch.tensor(np.transpose((arr - mean) / std, (2, 0, 1)), dtype=torch.float32)


def is_apple_or_droplet_object(img_gray):
    h, w = img_gray.shape
    _, mask = cv2.threshold(img_gray, 20, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return True
    c = max(contours, key=cv2.contourArea)
    x, y, bw, bh = cv2.boundingRect(c)
    roi = img_gray[y:y+bh, x:x+bw]
    top_m = float(roi[:int(bh*0.5), :].mean())
    bot_m = float(roi[int(bh*0.65):, :].mean())
    return (top_m > bot_m * 1.85)


def preprocess_with_metadata(img_bgr, target_size=(512, 512)):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    orig_h, orig_w = img_bgr.shape[:2]
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rx, ry, rw, rh = 0, 0, orig_w, orig_h
    if contours:
        c = max(contours, key=cv2.contourArea)
        if cv2.contourArea(c) > 1000:
            rx, ry, rw, rh = cv2.boundingRect(c)

    clahe = cv2.createCLAHE(clipLimit=1.2, tileGridSize=(16, 16))
    crop_rgb = cv2.cvtColor(clahe.apply(gray[ry:ry+rh, rx:rx+rw]), cv2.COLOR_GRAY2RGB)
    scale = min(target_size[0] / rw, target_size[1] / rh)
    nw, nh = max(1, int(rw * scale)), max(1, int(rh * scale))
    resized = cv2.resize(crop_rgb, (nw, nh), interpolation=cv2.INTER_AREA)

    canvas = np.zeros((target_size[1], target_size[0], 3), dtype=np.uint8)
    dx, dy = (target_size[0] - nw) // 2, (target_size[1] - nh) // 2
    canvas[dy:dy+nh, dx:dx+nw] = resized

    meta = {'rx': rx, 'ry': ry, 'rw': rw, 'rh': rh, 'scale': scale, 'nw': nw, 'nh': nh, 'dx': dx, 'dy': dy, 'orig_w': orig_w, 'orig_h': orig_h}
    return Image.fromarray(canvas), meta


def predict_with_4tta(model, pil_img):
    w, h = pil_img.size
    t_orig = safe_pil_to_tensor(pil_img).unsqueeze(0).to(device)
    t_flip = safe_pil_to_tensor(pil_img.transpose(Image.FLIP_LEFT_RIGHT)).unsqueeze(0).to(device)
    t_zoom_in = safe_pil_to_tensor(pil_img.resize((int(w * 1.05), int(h * 1.05))).crop((int(w * 0.025), int(h * 0.025), int(w * 0.025) + w, int(h * 0.025) + h))).unsqueeze(0).to(device)
    t_zoom_out = Image.new('RGB', (w, h), (0, 0, 0))
    t_zoom_out.paste(pil_img.resize((int(w * 0.95), int(h * 0.95))), (int(w * 0.025), int(h * 0.025)))
    t_zoom_out_tensor = safe_pil_to_tensor(t_zoom_out).unsqueeze(0).to(device)

    with torch.no_grad():
        o1 = F.softmax(model(t_orig), dim=1)
        o2 = F.softmax(model(t_flip), dim=1)
        o3 = F.softmax(model(t_zoom_in), dim=1)
        o4 = F.softmax(model(t_zoom_out_tensor), dim=1)
        avg_probs = (o1 + o2 + o3 + o4) / 4.0
    return avg_probs.squeeze(0), t_orig


def detect_mri_slice_orientation(gray, head_mask, meta):
    rx, ry, rw, rh = meta['rx'], meta['ry'], meta['rw'], meta['rh']
    vault_slice = head_mask[int(ry + 0.14 * rh):int(ry + 0.42 * rh), :]
    xs = np.where(vault_slice > 0)[1]
    if len(xs) > 20:
        cranial_w = max(60.0, float(np.percentile(xs, 98) - np.percentile(xs, 2)))
        cranial_cx = 0.5 * float(np.percentile(xs, 2) + np.percentile(xs, 98))
    else:
        cranial_w, cranial_cx = float(rw), float(rx + 0.5 * rw)

    vault_box_gray = gray[int(ry + 0.14 * rh):int(ry + 0.44 * rh), int(rx + 0.22 * rw):int(rx + 0.78 * rw)]
    vault_box_mask = head_mask[int(ry + 0.14 * rh):int(ry + 0.44 * rh), int(rx + 0.22 * rw):int(rx + 0.78 * rw)]
    vault_vals = vault_box_gray[(vault_box_mask > 0) & (vault_box_gray > 12)]
    vault_med = float(np.median(vault_vals)) if len(vault_vals) > 50 else 60.0

    air_thresh = max(15.0, min(44.0, vault_med * 0.52))
    solid_thresh = max(18.0, min(48.0, vault_med * 0.62))

    top_nasal_roi = gray[int(ry + 0.05 * rh):int(ry + 0.30 * rh), int(rx + 0.36 * rw):int(rx + 0.64 * rw)]
    bot_cereb_roi = gray[int(ry + 0.60 * rh):int(ry + 0.86 * rh), int(rx + 0.34 * rw):int(rx + 0.66 * rw)]
    top_dark_air = float(np.mean(top_nasal_roi < air_thresh)) if top_nasal_roi.size > 0 else 0.0
    bot_solid_brain = float(np.mean(bot_cereb_roi > solid_thresh)) if bot_cereb_roi.size > 0 else 0.0

    bot_center_strip = head_mask[int(ry + 0.90 * rh):int(ry + rh), int(rx + 0.25 * rw):int(rx + 0.75 * rw)]
    bot_center_fill = float(np.mean(bot_center_strip > 0)) if bot_center_strip.size > 0 else 0.0

    pharynx_col = gray[int(ry + 0.56 * rh):int(ry + 0.92 * rh), int(rx + 0.43 * rw):int(rx + 0.57 * rw)]
    left_jaw_col = gray[int(ry + 0.56 * rh):int(ry + 0.92 * rh), int(rx + 0.22 * rw):int(rx + 0.40 * rw)]
    right_jaw_col = gray[int(ry + 0.56 * rh):int(ry + 0.92 * rh), int(rx + 0.60 * rw):int(rx + 0.78 * rw)]
    pharynx_air = float(np.mean(pharynx_col < air_thresh)) if pharynx_col.size > 0 else 0.0
    has_coronal_pharynx = (pharynx_air > 0.35) and (float(np.mean(left_jaw_col > solid_thresh)) > 0.30) and (float(np.mean(right_jaw_col > solid_thresh)) > 0.30)

    is_axial_orbit = (not has_coronal_pharynx) and ((top_dark_air > 0.15 and bot_solid_brain > 0.68) or (top_dark_air > 0.14 and bot_solid_brain > 0.62 and bot_center_fill < 0.55))

    top_vault_roi = gray[int(ry + 0.10 * rh):int(ry + 0.36 * rh), int(rx + 0.28 * rw):int(rx + 0.72 * rw)]
    top_vault_solid = float(np.mean(top_vault_roi > solid_thresh)) if top_vault_roi.size > 0 else 0.0
    is_vertical_scan = (not is_axial_orbit) and (has_coronal_pharynx or (bot_center_fill > 0.42 and top_vault_solid > 0.62))

    roi_head = head_mask[ry:ry+rh, rx:rx+rw]
    head_sym_iou = float(np.sum((roi_head > 0) & (cv2.flip(roi_head, 1) > 0))) / max(1.0, float(np.sum((roi_head > 0) | (cv2.flip(roi_head, 1) > 0))))
    sagittal_facing = None
    if is_vertical_scan and (not has_coronal_pharynx) and (head_sym_iou < 0.84):
        left_m = head_mask[int(ry + 0.36 * rh):int(ry + 0.78 * rh), int(rx + 0.08 * rw):int(rx + 0.44 * rw)]
        right_m = head_mask[int(ry + 0.36 * rh):int(ry + 0.78 * rh), int(rx + 0.56 * rw):int(rx + 0.92 * rw)]
        left_air = float(np.mean(gray[int(ry + 0.36 * rh):int(ry + 0.78 * rh), int(rx + 0.08 * rw):int(rx + 0.44 * rw)][left_m > 0] < air_thresh)) if np.any(left_m > 0) else 0.0
        right_air = float(np.mean(gray[int(ry + 0.36 * rh):int(ry + 0.78 * rh), int(rx + 0.56 * rw):int(rx + 0.92 * rw)][right_m > 0] < air_thresh)) if np.any(right_m > 0) else 0.0
        if right_air > left_air + 0.10:
            sagittal_facing = "right"
        elif left_air > right_air + 0.10:
            sagittal_facing = "left"

    return {"cranial_w": cranial_w, "cranial_cx": cranial_cx, "air_thresh": air_thresh, "is_axial_orbit": is_axial_orbit, "is_vertical_scan": is_vertical_scan, "sagittal_facing": sagittal_facing}


def build_normalized_parenchyma_mask(gray, head_mask, meta, erode_ratio, predicted_class, is_dual_collage=False):
    orig_h, orig_w = gray.shape
    rx, ry, rw, rh = meta['rx'], meta['ry'], meta['rw'], meta['rh']
    orient = detect_mri_slice_orientation(gray, head_mask, meta)
    erode_px = max(5, int(min(rw, rh) * erode_ratio))
    parenchyma = cv2.erode(head_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_px, erode_px)), iterations=1)

    if not is_dual_collage and predicted_class == "Глиома":
        y_grid, x_grid = np.mgrid[0:orig_h, 0:orig_w]
        if orient["is_axial_orbit"]:
            parenchyma[:int(ry + 0.33 * rh), :] = 0
            parenchyma[:int(ry + 0.46 * rh), max(0, int(orient["cranial_cx"] - 0.20 * orient["cranial_w"])):min(orig_w, int(orient["cranial_cx"] + 0.20 * orient["cranial_w"]))] = 0
        elif orient["is_vertical_scan"]:
            if orient["sagittal_facing"] == "right":
                parenchyma[int(ry + 0.60 * rh):, :] = 0
                parenchyma[int(ry + 0.27 * rh):, int(rx + 0.58 * rw):] = 0
            elif orient["sagittal_facing"] == "left":
                parenchyma[int(ry + 0.60 * rh):, :] = 0
                parenchyma[int(ry + 0.27 * rh):, :int(rx + 0.42 * rw)] = 0
            else:
                parenchyma[int(ry + 0.53 * rh):, :] = 0
                dome_mask = (((x_grid - orient["cranial_cx"]) / max(25.0, 0.43 * orient["cranial_w"])) ** 2 + ((y_grid - (ry + 0.30 * rh)) / max(25.0, 0.235 * rh)) ** 2) <= 1.0
                parenchyma[~dome_mask] = 0
                parenchyma[int(ry + 0.44 * rh):, max(0, int(orient["cranial_cx"] - 0.18 * orient["cranial_w"])):min(orig_w, int(orient["cranial_cx"] + 0.18 * orient["cranial_w"]))] = 0
        else:
            parenchyma[:int(ry + 0.18 * rh), max(0, int(orient["cranial_cx"] - 0.15 * orient["cranial_w"])):min(orig_w, int(orient["cranial_cx"] + 0.15 * orient["cranial_w"]))] = 0

    return parenchyma, orient


def segment_glioma_with_z_level_normalization(gray, parenchyma_mask, cam_masked, meta, profile, orient, is_dual_collage=False, min_area=40):
    orig_h, orig_w = gray.shape
    rx, ry, rw, rh = meta['rx'], meta['ry'], meta['rw'], meta['rh']
    smooth = cv2.bilateralFilter(gray, 7, 30, 30).astype(np.float32)
    brain_pixels = smooth[(parenchyma_mask > 0) & (gray > max(14.0, orient.get("air_thresh", 18.0) * 0.75))]
    if len(brain_pixels) < 100:
        return []

    med_val = float(np.median(brain_pixels))
    std_val = max(5.5, float((np.percentile(brain_pixels, 75) - np.percentile(brain_pixels, 25)) * 0.7413))
    z_map = (smooth - med_val) / std_val

    is_symmetric_view = (orient["sagittal_facing"] is None) and (not is_dual_collage)
    z_asym = np.zeros_like(z_map)
    if is_symmetric_view:
        cx_int = int(round(orient["cranial_cx"]))
        blur_z = cv2.GaussianBlur(z_map, (21, 21), 0)
        flipped_z = np.zeros_like(blur_z)
        for x_col in range(rx, min(orig_w, rx + rw)):
            mirror_x = 2 * cx_int - x_col
            if 0 <= mirror_x < orig_w:
                flipped_z[:, x_col] = blur_z[:, mirror_x]
        z_asym = np.abs(blur_z - flipped_z) * (parenchyma_mask > 0).astype(np.float32)
    else:
        z_asym = np.abs(z_map) * (parenchyma_mask > 0).astype(np.float32)

    inner_core_mask = (cv2.distanceTransform(parenchyma_mask, cv2.DIST_L2, 5) >= max(3.0, min(rw, rh) * 0.022)).astype(np.uint8) * 255

    close_ring_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    bright_closed = cv2.morphologyEx(((z_map >= profile["z_bright_thresh"]) & (inner_core_mask > 0)).astype(np.uint8) * 255, cv2.MORPH_CLOSE, close_ring_k)
    ring_cnts, _ = cv2.findContours(bright_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bright_filled = np.zeros_like(gray)
    for rc in ring_cnts:
        if cv2.contourArea(rc) >= min_area:
            cv2.drawContours(bright_filled, [rc], -1, 255, thickness=cv2.FILLED)
    bright_binary = cv2.morphologyEx(cv2.bitwise_and(bright_filled, inner_core_mask), cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))

    dark_raw = ((z_map <= profile["z_dark_thresh"]) & (gray >= 14) & (inner_core_mask > 0)).astype(np.uint8) * 255
    dark_binary = cv2.morphologyEx(cv2.morphologyEx(dark_raw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))), cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))

    bright_dilated = cv2.dilate(bright_binary, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)), iterations=1)
    dark_dilated = cv2.dilate(dark_binary, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)), iterations=1)

    composite_binary = np.zeros_like(gray)
    num_d, labels_d = cv2.connectedComponents(dark_binary)
    for lbl in range(1, num_d):
        comp = (labels_d == lbl).astype(np.uint8) * 255
        if np.any((comp > 0) & (bright_dilated > 0)):
            composite_binary = cv2.bitwise_or(composite_binary, comp)

    num_b, labels_b = cv2.connectedComponents(bright_binary)
    for lbl in range(1, num_b):
        comp = (labels_b == lbl).astype(np.uint8) * 255
        if np.any((comp > 0) & (dark_dilated > 0)):
            composite_binary = cv2.bitwise_or(composite_binary, comp)

    if np.any(composite_binary > 0):
        composite_binary = cv2.morphologyEx(composite_binary, cv2.MORPH_CLOSE, close_ring_k)

    brain_area = max(1.0, float(np.sum(parenchyma_mask > 0)))
    candidates = []

    for mode, bin_img in [("composite", composite_binary), ("bright", bright_binary), ("dark", dark_binary)]:
        if not np.any(bin_img > 0):
            continue
        cnts, _ = cv2.findContours(bin_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in cnts:
            c_mask = np.zeros_like(gray)
            cv2.drawContours(c_mask, [c], -1, 255, thickness=cv2.FILLED)
            c_mask = cv2.bitwise_and(c_mask, inner_core_mask)
            area = float(np.sum(c_mask > 0))

            if area < min_area or (area / brain_area) > profile["max_area_ratio"]:
                continue

            if mode in ("dark", "composite"):
                c_mask = cv2.bitwise_and(cv2.dilate(c_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1), parenchyma_mask)
                ref_cnts, _ = cv2.findContours(c_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if not ref_cnts:
                    continue
                c = max(ref_cnts, key=cv2.contourArea)
                area = float(cv2.contourArea(c))
                if area < min_area or (area / brain_area) > profile["max_area_ratio"]:
                    continue

            solidity = area / max(1.0, float(cv2.contourArea(cv2.convexHull(c))))
            cx, cy, cw_box, ch_box = cv2.boundingRect(c)
            (_, _), (rot_w, rot_h), _ = cv2.minAreaRect(c)
            elongation = max(max(cw_box, ch_box) / max(1.0, min(cw_box, ch_box)), max(rot_w, rot_h) / max(1.0, min(rot_w, rot_h)))

            mean_cam = float(cam_masked[c_mask > 0].mean())
            max_cam = float(cam_masked[c_mask > 0].max())
            mean_asym = float(z_asym[c_mask > 0].mean())
            mean_z_mag = float(np.abs(z_map[c_mask > 0]).mean())

            if max_cam < 0.22 or mean_cam < 0.12 or elongation > 3.2 or (elongation > 2.4 and solidity < 0.62):
                continue
            if is_symmetric_view and mode == "dark" and mean_asym < 0.52 and (abs((cx + 0.5 * cw_box) - orient["cranial_cx"]) / max(1.0, orient["cranial_w"]) < 0.12 or max_cam < 0.45):
                continue

            score = (area ** 0.58) * ((0.20 + 0.80 * max_cam) * (0.25 + 0.75 * mean_cam)) * (0.45 * min(2.5, mean_z_mag) + 0.55 * min(2.5, mean_asym)) * (0.35 + 0.65 * (solidity ** 1.2)) * (0.35 if (elongation > 2.2 and solidity < 0.68) else 1.0) * (1.38 if mode == "composite" else 1.0)
            candidates.append((score, c))

    if not candidates:
        return []
    candidates.sort(key=lambda x: x[0], reverse=True)
    return [candidates[0][1]]


def segment_pituitary_with_normalized_level(gray, head_mask, cam_masked, meta, min_area=25):
    orig_h, orig_w = gray.shape
    rx, ry, rw, rh = meta['rx'], meta['ry'], meta['rw'], meta['rh']
    orient = detect_mri_slice_orientation(gray, head_mask, meta)
    sella_cx = orient["cranial_cx"]

    sinus_col = gray[int(ry + 0.50 * rh):int(ry + 0.75 * rh), max(0, int(sella_cx - 0.08 * rw)):min(orig_w, int(sella_cx + 0.08 * rw))]
    dark_rows = np.where(np.mean(sinus_col < 45, axis=1) > 0.35)[0]
    sella_cy = np.clip((float(int(ry + 0.50 * rh) + dark_rows[0]) - 0.015 * rh) if len(dark_rows) > 0 else (ry + 0.595 * rh), ry + 0.555 * rh, ry + 0.645 * rh)

    tophat_nodule = cv2.morphologyEx(cv2.GaussianBlur(gray, (5, 5), 0), cv2.MORPH_TOPHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (max(15, int(min(rw, rh) * 0.105)), max(15, int(min(rw, rh) * 0.105))))).astype(np.float32)
    if tophat_nodule.max() > 0:
        tophat_nodule /= tophat_nodule.max()

    y_grid, x_grid = np.mgrid[0:orig_h, 0:orig_w]
    gauss = np.exp(-(((x_grid - sella_cx) ** 2) / (2 * (0.055 * rw) ** 2) + ((y_grid - sella_cy) ** 2) / (2 * (0.042 * rh) ** 2)))
    sella_box = ((x_grid >= sella_cx - 0.09 * rw) & (x_grid <= sella_cx + 0.09 * rw) & (y_grid >= sella_cy - 0.055 * rh) & (y_grid <= sella_cy + 0.065 * rh)).astype(np.float32)
    pit_map = gauss * (0.35 + 0.65 * tophat_nodule) * np.clip((cv2.GaussianBlur(gray, (5, 5), 0).astype(np.float32) - 52.0) / 28.0, 0.0, 1.0) * sella_box
    if pit_map.max() > 0:
        pit_map /= pit_map.max()

    raw_mask = cv2.morphologyEx(cv2.morphologyEx((pit_map >= 0.42).astype(np.uint8) * 255, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))), cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    cnts, _ = cv2.findContours(raw_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = [c for c in cnts if cv2.contourArea(c) >= min_area]
    return [max(valid, key=cv2.contourArea)] if valid else []


def expand_meningioma_region(seed_mask, gray, head_mask):
    if np.sum(seed_mask > 0) < 40:
        return seed_mask
    orig_h, orig_w = gray.shape
    smooth_gray = cv2.bilateralFilter(gray, 7, 35, 35)
    seed_pixels = smooth_gray[seed_mask > 0]
    brain_pixels = smooth_gray[head_mask > 0]
    if float(np.median(seed_pixels)) < float(np.median(brain_pixels)) + 0.25 * float(np.std(brain_pixels)):
        return seed_mask

    allowed_zone = cv2.dilate(seed_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (max(15, int(min(orig_h, orig_w) * 0.14)), max(15, int(min(orig_h, orig_w) * 0.14)))), iterations=1)
    candidate_mask = cv2.morphologyEx(cv2.morphologyEx(cv2.bitwise_and(((smooth_gray >= max(float(np.median(brain_pixels)) + 0.35 * float(np.std(brain_pixels)), float(np.percentile(seed_pixels, 20)) - 8.0)) & (head_mask > 0)).astype(np.uint8) * 255, allowed_zone), cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))), cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))

    num_labels, labels = cv2.connectedComponents(candidate_mask)
    expanded = seed_mask.copy()
    for lbl in range(1, num_labels):
        comp = (labels == lbl).astype(np.uint8) * 255
        if np.sum((comp > 0) & (seed_mask > 0)) > 0 and np.sum(comp > 0) <= (np.sum(head_mask > 0) * 0.28):
            expanded = cv2.bitwise_or(expanded, comp)
    return expanded


def generate_segmentation_mask_and_annotation(image_path, min_contour_area=40):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return None
    orig_h, orig_w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    if float(np.mean((np.abs(img_bgr[:, :, 0].astype(int) - img_bgr[:, :, 1].astype(int)) > 15) | (np.abs(img_bgr[:, :, 1].astype(int) - img_bgr[:, :, 2].astype(int)) > 15))) > 0.08:
        return {"status": "rejected", "reason": "Обнаружено цветное изображение. Система калибрована только для монохромных срезов МРТ."}

    canvas_pil, meta = preprocess_with_metadata(img_bgr, target_size=(512, 512))
    diag_probs, tensor = predict_with_4tta(model_diag, canvas_pil)

    not_mri_prob = float(diag_probs[3].item()) if num_classes == 5 else 0.0
    if use_gatekeeper:
        gate_probs, _ = predict_with_4tta(model_gate, canvas_pil)
        not_mri_prob = max(not_mri_prob, float(gate_probs[0].item()))

    is_apple = is_apple_or_droplet_object(gray)
    if not_mri_prob > 0.50 or is_apple:
        conf_val = max(not_mri_prob, 0.96 if is_apple else not_mri_prob)
        return {"status": "not_mri", "class": "Не МРТ", "confidence": float(conf_val), "probabilities": {"Не МРТ": float(conf_val), "МРТ мозга": float(1.0 - conf_val)}}

    med_idx_map = [0, 1, 2, 4] if num_classes == 5 else [0, 1, 2, 3]
    med_names = ['Гипофиз', 'Глиома', 'Менингиома', 'Опухоли нет']
    med_probs = diag_probs[med_idx_map] / (diag_probs[med_idx_map].sum() + 1e-7) if num_classes == 5 else diag_probs

    conf, local_idx = torch.max(med_probs, 0)
    predicted_class = med_names[local_idx.item()]
    target_cam_class = med_idx_map[local_idx.item()]

    if float(med_probs[3].item()) >= NOTUMOR_CONFIDENCE_FLOOR:
        predicted_class, conf, target_cam_class = "Опухоли нет", med_probs[3], med_idx_map[3]
    elif predicted_class == "Гипофиз" and float(med_probs[0].item()) < 0.60 and float(med_probs[1].item()) > 0.30:
        predicted_class, conf, target_cam_class = "Глиома", med_probs[1], med_idx_map[1]

    is_tumor = (predicted_class != "Опухоли нет")
    annotated_img = img_rgb.copy()
    binary_mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
    polygons, bounding_boxes = [], []

    if is_tumor:
        profile = CLASS_SEG_PROFILES[predicted_class]
        cam = grad_cam_engine.get_cam(tensor, int(target_cam_class), w_l4=profile["cam_l4_weight"], w_l3=profile["cam_l3_weight"])
        cam_full = np.zeros((orig_h, orig_w), dtype=np.float32)
        cam_full[meta['ry']:meta['ry']+meta['rh'], meta['rx']:meta['rx']+meta['rw']] = cv2.resize(cv2.resize(cam, (512, 512))[meta['dy']:meta['dy']+meta['nh'], meta['dx']:meta['dx']+meta['nw']], (meta['rw'], meta['rh']))

        head_mask = np.zeros_like(gray)
        cnts, _ = cv2.findContours(cv2.threshold(gray, 18, 255, cv2.THRESH_BINARY)[1], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_heads = [hc for hc in cnts if cv2.contourArea(hc) > (orig_h * orig_w * 0.05)]
        if valid_heads:
            cv2.drawContours(head_mask, valid_heads, -1, 255, thickness=cv2.FILLED)

        parenchyma_mask, orient = build_normalized_parenchyma_mask(gray, head_mask, meta, profile["skull_erode_ratio"], predicted_class, is_dual_collage=(len(valid_heads) >= 2 or meta['rw'] > meta['rh'] * 1.18))
        cam_masked = cam_full * (parenchyma_mask / 255.0)
        if cam_masked.max() > 0:
            cam_masked /= cam_masked.max()

        selected_contours = []
        if predicted_class == "Гипофиз":
            selected_contours = segment_pituitary_with_normalized_level(gray, head_mask, cam_masked, meta, min_area=25)
        elif predicted_class == "Глиома":
            selected_contours = segment_glioma_with_z_level_normalization(gray, parenchyma_mask, cam_masked, meta, profile, orient, min_area=min_contour_area)

        if not selected_contours:
            active = (cam_masked * 255).astype(np.uint8)
            nonzero = active[active > 20]
            if len(nonzero) > 45:
                otsu_thr, _ = cv2.threshold(nonzero, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                f_thr = min(0.76, max(profile.get("min_thresh", 0.38), (otsu_thr / 255.0) * profile.get("otsu_multiplier", 0.92)))
                c_mask = cv2.morphologyEx(cv2.morphologyEx(((cam_masked >= f_thr) & (parenchyma_mask > 0)).astype(np.uint8) * 255, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))), cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
                if predicted_class == "Менингиома":
                    c_mask = expand_meningioma_region(c_mask, gray, head_mask)
                res_cnts = [c for c in cv2.findContours(c_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0] if cv2.contourArea(c) >= min_contour_area]
                if res_cnts:
                    selected_contours = [max(res_cnts, key=cv2.contourArea)]

        if selected_contours:
            cv2.drawContours(binary_mask, selected_contours, -1, 255, thickness=cv2.FILLED)
            color_mask = np.zeros_like(img_rgb)
            color_mask[binary_mask == 255] = [230, 40, 40]
            annotated_img = cv2.addWeighted(annotated_img, 0.70, color_mask, 0.30, 0)
            for c in selected_contours:
                cv2.drawContours(annotated_img, [c], -1, (255, 30, 30), 2)
                x, y, w, h = cv2.boundingRect(c)
                cv2.rectangle(annotated_img, (x, y), (x + w, y + h), (255, 30, 30), 2)
                bounding_boxes.append({"x1": int(x), "y1": int(y), "x2": int(x + w), "y2": int(y + h)})
                polygons.append(c.reshape(-1, 2).tolist())

    return {
        "status": "success",
        "class": predicted_class,
        "confidence": float(conf.item()),
        "is_tumor": is_tumor,
        "annotated_image": annotated_img,
        "binary_mask": binary_mask,
        "annotation_json": {
            "file_name": os.path.basename(image_path),
            "predicted_class": predicted_class,
            "confidence": float(conf.item()),
            "tumor_detected": is_tumor,
            "image_size": {"height": int(orig_h), "width": int(orig_w)},
            "bounding_boxes": bounding_boxes,
            "segmentation_polygons": polygons
        },
        "probabilities": {med_names[i]: float(med_probs[i].item()) for i in range(len(med_names))}
    }


def format_output(result, fallback_img):
    if result is None:
        return fallback_img, "Ошибка чтения файла", "{}"
    if result["status"] == "rejected":
        return fallback_img, f"Отклонено:\n{result['reason']}", "{}"
    if result["status"] == "not_mri":
        prob_str = "\n".join([f"{k}: {v*100:.2f}%" for k, v in result["probabilities"].items()])
        return fallback_img, f"Статус: Не МРТ\nУверенность входного фильтра: {result['confidence']*100:.1f}%\n\n{prob_str}", "{}"

    prob_str = "\n".join([f"{k}: {v*100:.2f}%" for k, v in result["probabilities"].items()])
    diagnosis_output = f"Класс патологии: {result['class']}\nУверенность классификатора: {result['confidence']*100:.1f}%\n\nРаспределение вероятностей:\n{prob_str}"
    return result["annotated_image"], diagnosis_output, json.dumps(result["annotation_json"], indent=2, ensure_ascii=False)


def process_file_upload(input_image):
    if input_image is None:
        return None, "Файл не предоставлен", "{}"
    try:
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
        temp_file.close()
        cv2.imwrite(temp_file.name, cv2.cvtColor(np.asarray(input_image, dtype=np.uint8), cv2.COLOR_RGB2BGR))
        result = generate_segmentation_mask_and_annotation(temp_file.name)
        if os.path.exists(temp_file.name):
            os.remove(temp_file.name)
        return format_output(result, input_image)
    except Exception as e:
        return input_image, f"Техническая ошибка:\n{str(e)}\n{traceback.format_exc()}", "{}"


def process_url_input(url):
    url = url.strip()
    if not url:
        return None, "URL-адрес не указан", "{}"
    try:
        resp = requests.get(url, timeout=12, headers={'User-Agent': 'Mozilla/5.0'})
        if resp.status_code != 200:
            return None, f"Ошибка загрузки: статус HTTP {resp.status_code}", "{}"
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".jpg")
        with open(temp_file.name, "wb") as f:
            f.write(resp.content)
        result = generate_segmentation_mask_and_annotation(temp_file.name)
        raw_bgr = cv2.imread(temp_file.name)
        raw_np = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB) if raw_bgr is not None else None
        if os.path.exists(temp_file.name):
            os.remove(temp_file.name)
        return format_output(result, raw_np)
    except Exception as e:
        return None, f"Ошибка обработки URL: {str(e)}", "{}"


with gr.Blocks(title="СППР: Анализ МРТ головного мозга") as demo:
    gr.Markdown("### Система поддержки принятия врачебных решений: Анализ МРТ головного мозга")
    with gr.Row():
        with gr.Column():
            with gr.Tab("Загрузка с устройства"):
                file_input = gr.Image(type="numpy", label="Выбор файла МРТ")
                btn_file = gr.Button("Анализировать файл", variant="primary")
            with gr.Tab("Загрузка по ссылке"):
                url_input = gr.Textbox(placeholder="Прямая ссылка на срез (.jpg, .png)", label="URL")
                btn_url = gr.Button("Загрузить и анализировать", variant="primary")
        with gr.Column():
            out_img = gr.Image(type="numpy", label="Результат локализации и сегментации")
            out_diag = gr.Textbox(label="Заключение классификатора", lines=7)
            out_json = gr.Code(label="Предварительная разметка (JSON)", language="json")

    btn_file.click(fn=process_file_upload, inputs=[file_input], outputs=[out_img, out_diag, out_json])
    btn_url.click(fn=process_url_input, inputs=[url_input], outputs=[out_img, out_diag, out_json])

demo.launch(share=True, debug=False)

Downloading...
From (original): https://drive.google.com/uc?id=18C2quutj5WNqhHAfSm_UKlxlqcNyuid9
From (redirected): https://drive.google.com/uc?id=18C2quutj5WNqhHAfSm_UKlxlqcNyuid9&confirm=t&uuid=093bffa6-3cdc-4a32-bb63-3c5aeb17aecd
To: /content/best_resnet50_latest.pth
100%|██████████| 94.4M/94.4M [00:00<00:00, 159MB/s]


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5509528f356547c1e5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
